# Как выглядит сигнал приплюснутой планеты и её спутника в транзитной кривой блеска

Иллюстрация к статье Le-Chris Wang & Joshua N. Winn (2026),
*"How Many Transiting Giant Planets Can JWST Search for Moons and Rotational Oblateness?"*,
The Astrophysical Journal Letters, arXiv:10.3847/2041-8213/ae89a6 (разделы 3.1 «Oblateness» и 3.2 «Satellites»).

Статья спрашивает: сколько экзопланет-гигантов вообще позволяют телескопу JWST заметить, что планета
не идеально круглая (приплюснута из-за быстрого вращения, как Юпитер и Сатурн) или что у неё есть спутник?
Но на словах этот сигнал понять трудно — гораздо нагляднее увидеть его своими глазами на настоящей,
пусть и упрощённой, модели транзита.

Здесь мы строим **точную геометрическую модель перекрытия** диска звезды силуэтом планеты (через библиотеку
`shapely` — без приближений методом сетки), используя реальные числа: приплюснутость Юпитера ≈6.5% (значение,
которое приводит сама статья), оптимальный для обнаружения прицельный параметр транзита b=1/√2≈0.71 (тоже
собственная рекомендация статьи, раздел 3.1), и отношение радиусов Ганимед/Юпитер ≈0.038 для модели спутника.

**Важная оговорка** (по итогам консультации с Fellow_Astrophysicist, см.
`10.3847_2041-8213_ae89a6_viz_consult_note.md` в папке статьи): модель без потемнения к краю звезды — честное
педагогическое упрощение, не искажающее саму асимметрию входа-выхода (она чисто геометрическая). Но реальный
сигнал, который ищет JWST, — это 10–100 миллионных долей (ppm) яркости звезды: тонкая деталь на грани
чувствительности приборов, а не что-то заметное невооружённым глазом на настоящих данных. Ниже мы честно
подписываем реальный масштаб сигнала в ppm, даже когда для наглядности используем увеличенный радиус планеты.

In [1]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from shapely.geometry import Point
from shapely.affinity import scale, rotate, translate

OUT_DIR = Path("figures")
OUT_DIR.mkdir(exist_ok=True)

THEME = "dark"  # "light" or "dark" -- controls the color scheme of all plots below

if THEME == "dark":
    plt.style.use("dark_background")
    FG_COLOR = "white"
elif THEME == "light":
    plt.style.use("default")
    FG_COLOR = "black"
else:
    raise ValueError(f"THEME must be 'light' or 'dark', got {THEME!r}")

def occulter_polygon(cx, cy, rx, ry, theta_deg, res=96):
    # a circle/ellipse silhouette (rx=ry gives a plain circle) placed at (cx, cy) in stellar radii
    base = Point(0, 0).buffer(1.0, resolution=res)
    e = scale(base, rx, ry, origin=(0, 0))
    e = rotate(e, theta_deg, origin=(0, 0), use_radians=False)
    e = translate(e, cx, cy)
    return e

R_STAR = 1.0
star = Point(0, 0).buffer(R_STAR, resolution=400)
STAR_AREA = np.pi * R_STAR**2

ModuleNotFoundError: No module named 'shapely'

## Сигнал приплюснутости

Приплюснутая планета проходит по диску звезды не идеальным кругом, а слегка вытянутым эллипсом — из-за этого
вход и выход из транзита занимают чуть разное время, и появляется едва заметная асимметрия. Чтобы увидеть её
надёжно, строим не саму кривую блеска (её форма почти неотличима от сферической планеты на глаз), а **разницу**
между приплюснутой и сферической моделями — именно так реальный сигнал и ищут в данных.

In [ ]:
R_PLANET = 0.15          # planet-to-star radius ratio, typical hot-Jupiter scale (visual clarity)
IMPACT_B = 1 / np.sqrt(2)  # paper's own optimal impact parameter for oblateness detection (Section 3.1)
F_OBLATE = 0.065           # Jupiter's actual oblateness, quoted in the paper
THETA_PERP = 45.0          # projected orientation of the flattening axis vs the transit chord

area_target = np.pi * R_PLANET**2
rx, ry = R_PLANET, R_PLANET * (1 - F_OBLATE)
scale_fac = np.sqrt(area_target / (np.pi * rx * ry))  # keep the same silhouette area as the spherical case
rx *= scale_fac
ry *= scale_fac

xs = np.linspace(-1.1, 1.1, 440)
F_sphere = np.empty_like(xs)
F_oblate = np.empty_like(xs)
for i, x in enumerate(xs):
    occ_sphere = occulter_polygon(x, IMPACT_B, R_PLANET, R_PLANET, 0.0)
    occ_oblate = occulter_polygon(x, IMPACT_B, rx, ry, THETA_PERP)
    F_sphere[i] = 1 - star.intersection(occ_sphere).area / STAR_AREA
    F_oblate[i] = 1 - star.intersection(occ_oblate).area / STAR_AREA

depth_ppm = (1 - F_sphere.min()) * 1e6
resid_ppm = (F_oblate - F_sphere) * 1e6

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7.5, 7), sharex=True, height_ratios=[2, 1])
ax1.plot(xs, F_sphere, color="#1f77b4", lw=2, label="Сферическая планета")
ax1.plot(xs, F_oblate, color="#d62728", lw=2, ls="--", label="Приплюснутая (как Юпитер)")
ax1.set_ylabel("Относительный блеск звезды")
ax1.set_title(f"Транзит, b={IMPACT_B:.2f} -- глубина ~{depth_ppm:,.0f} ppm (масштаб реалистичный)".replace(",", " "))
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(xs, resid_ppm, color="#2ca02c", lw=2)
ax2.axhline(0, color=FG_COLOR, lw=0.7)
ax2.set_xlabel("Смещение планеты по диску звезды [радиусы звезды]")
ax2.set_ylabel("Приплюснутая − сферическая\n[ppm]")
ax2.set_title(f"Сигнал приплюснутости: пик до {np.max(np.abs(resid_ppm)):.0f} ppm,\nантисимметричен относительно середины транзита", fontsize=10)
ax2.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUT_DIR / "oblateness_transit_signal.png", dpi=140)
plt.show()

print(f"Глубина транзита: {depth_ppm:.0f} ppm")
print(f"Пиковый сигнал приплюснутости (разница): {np.max(np.abs(resid_ppm)):.0f} ppm")
print("Для сравнения: статья говорит о сигнале порядка 10-100 ppm при реалистичном (не увеличенном) радиусе планеты.")

## Сигнал спутника

Спутник выдаёт себя иначе — не искажением формы транзита самой планеты, а отдельным, гораздо более мелким
провалом блеска, смещённым по времени. Возьмём спутник размером с Ганимед (крупнейший спутник Юпитера) —
отношение радиусов Ганимед/Юпитер ≈0.038, подтверждено консультацией.

In [ ]:
R_MOON = R_PLANET * 0.038  # Ganymede/Jupiter radius ratio
MOON_OFFSET_B = IMPACT_B + 0.22  # moon on a slightly different chord, offset from the planet's

xs_wide = np.linspace(-1.6, 2.0, 700)
F_planet = np.empty_like(xs_wide)
F_moon_only = np.empty_like(xs_wide)
for i, x in enumerate(xs_wide):
    occ_p = occulter_polygon(x, IMPACT_B, R_PLANET, R_PLANET, 0.0)
    occ_m = occulter_polygon(x - 1.35, MOON_OFFSET_B, R_MOON, R_MOON, 0.0)  # moon transits ~1.35 (star radii) later
    F_planet[i] = 1 - star.intersection(occ_p).area / STAR_AREA
    F_moon_only[i] = 1 - star.intersection(occ_m).area / STAR_AREA

F_combined = F_planet - (1 - F_moon_only)  # both dips subtract flux; combine additively in depth

fig, (axL, axR) = plt.subplots(1, 2, figsize=(10.5, 4.5), width_ratios=[2, 1])
axL.plot(xs_wide, F_combined, color="#1f77b4", lw=2)
axL.set_xlabel("Время [условные единицы,\nширина транзита планеты ~1]")
axL.set_ylabel("Относительный блеск звезды")
axL.set_title("Полная кривая -- провал\nспутника не виден в этом масштабе", fontsize=10)
axL.grid(alpha=0.3)
axL.axvspan(0.9, 1.8, color="orange", alpha=0.15)

mask = (xs_wide > 0.85) & (xs_wide < 1.85)
axR.plot(xs_wide[mask], F_moon_only[mask], color="#ff7f0e", lw=2)
axR.set_xlabel("Время (увеличено)")
axR.set_ylabel("Блеск (увеличенный масштаб)")
axR.set_title(f"Только провал от спутника\n(глубина ~{(1-F_moon_only.min())*1e6:.0f} ppm)", fontsize=10)
axR.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUT_DIR / "moon_transit_signal.png", dpi=140)
plt.show()

moon_depth_ppm = (1 - F_moon_only.min()) * 1e6
planet_depth_ppm = (1 - F_planet.min()) * 1e6
print(f"Глубина провала планеты: {planet_depth_ppm:.0f} ppm")
print(f"Глубина провала спутника: {moon_depth_ppm:.0f} ppm (в {planet_depth_ppm/moon_depth_ppm:.0f} раз мельче)")

## Итог

Оба графика выше показывают, почему статья вообще ставит вопрос "а хватит ли чувствительности JWST" —
и приплюснутость, и спутник дают сигналы, которые в реальных, не увеличенных для наглядности масштабах
измеряются считаными десятками миллионных долей яркости звезды. Это не то, что можно заметить на глаз —
такой сигнал нужно statистически выцарапывать из шумной кривой блеска, сравнивая её с точной моделью
идеальной сферы (именно поэтому наша нижняя панель "приплюснутая минус сферическая" — это, по сути, то,
что реально ищут в данных). Авторы статьи посчитали, для скольких из уже известных и ещё не открытых планет
таких размеров и такой яркости звезды-хозяина этот тонкий сигнал вообще в принципе достижим при разумном
уровне шума JWST — и получили обнадёживающий, но не безграничный список кандидатов.